# Experiment: MinecraftClaw Raw Export Profiling

Objective:
- Profile the first real MinecraftClaw raw export.
- Quantify noise sources before designing the cleaned MCP schema.
- Produce cleaning rules that reduce token cost without losing build-critical context.


In [ ]:
from __future__ import annotations

import json
from collections import Counter
from pathlib import Path
from statistics import fmean

EXPORT_DIR = Path(r"/Users/dracoglasser/Library/Application Support/PrismLauncher/instances/乌托邦探险之旅3.5fix/minecraft/minecraftclaw/exports/20260404T094308Z-glasserrrr")
FILES = [
    "scan_meta.json",
    "player_state.json",
    "inventory.json",
    "local_blocks.json",
    "block_entities.json",
    "entities.json",
    "surface_map.json",
]

raw = {file_name: json.loads((EXPORT_DIR / file_name).read_text()) for file_name in FILES}
{file_name: (len(value) if isinstance(value, list) else sorted(value.keys())) for file_name, value in raw.items()}


## Plan

- Measure raw volume and identify the noisiest fields.
- Check whether local structure decisions need raw coordinates or only summaries.
- Define a first-pass cleaned schema for the MCP server.


In [ ]:
local_blocks = raw["local_blocks.json"]
surface_map = raw["surface_map.json"]
entities = raw["entities.json"]
block_entities = raw["block_entities.json"]
inventory = raw["inventory.json"]

non_air_blocks = [entry for entry in local_blocks if not entry["is_air"]]
air_ratio = sum(1 for entry in local_blocks if entry["is_air"]) / len(local_blocks)
surface_heights = [entry["surface_y"] for entry in surface_map]
non_empty_inventory = [entry for entry in inventory if not entry["empty"]]

profile = {
    "local_block_count": len(local_blocks),
    "non_air_block_count": len(non_air_blocks),
    "air_ratio": round(air_ratio, 4),
    "top_non_air_blocks": Counter(entry["block_id"] for entry in non_air_blocks).most_common(12),
    "surface_sample_count": len(surface_map),
    "surface_height": {
        "min": min(surface_heights),
        "max": max(surface_heights),
        "range": max(surface_heights) - min(surface_heights),
        "average": round(fmean(surface_heights), 2),
    },
    "top_surface_blocks": Counter(entry["block_id"] for entry in surface_map).most_common(12),
    "entities_by_type": Counter(entry["type"] for entry in entities).most_common(),
    "block_entities_by_block": Counter(entry["block_id"] for entry in block_entities).most_common(),
    "non_empty_inventory": non_empty_inventory,
}
profile


In [ ]:
heavy_rows = {
    "entity_sample": entities[0],
    "block_entity_sample": block_entities[0],
    "inventory_sample": inventory[0],
}
heavy_rows


## Results

Key observations:
- `local_blocks.json` is dominated by air, so raw voxel export is too expensive for direct agent use.
- `surface_map.json` is much cheaper than full local blocks and is a good base for site selection.
- `entities.json` and `block_entities.json` are low-count but contain very heavy SNBT blobs.
- `block_entities.json.type` is not stable enough to keep as a public field; the SNBT `id` or `block_id` is more useful.

First-pass cleaning rules:
- Drop air entries from local block summaries and keep only aggregate counts plus thin y-level profiles.
- Replace raw SNBT with compact identifiers and positions in cleaned summaries.
- Keep full player pose and health state.
- Keep only non-empty inventory stacks plus totals-by-item.
- Keep surface height stats and dominant surface blocks instead of the full 32x32 grid in the main summary.


In [ ]:
cleaned_schema_outline = {
    "context": ["rawSchemaVersion", "cleanedSchemaVersion", "scannedAt", "dimension", "center", "localBox", "surfaceBox"],
    "player": ["name", "uuid", "blockPos", "exactPos", "yaw", "pitch", "health", "food", "saturation", "onGround", "hands"],
    "inventory": ["totalSlots", "occupiedSlots", "stacks", "totalsByItem"],
    "localEnvironment": ["totalBlocks", "airBlocks", "nonAirBlocks", "airRatio", "topNonAirBlocks", "yLevelProfiles"],
    "surface": ["sampleCount", "height", "topBlocks"],
    "entities": ["total", "byType", "nearby"],
    "pointsOfInterest": ["total", "byBlockId", "entries"],
}
cleaned_schema_outline


## Next steps

- Run the MCP-side cleaner on more export samples from different terrain types.
- Decide whether the final agent tool should return one compact summary or a compact summary plus optional detail slices.
- Add flat-area and buildability heuristics only after validating this first-pass schema on multiple worlds.
